In [0]:
# Create widgets
dbutils.widgets.text("source_system", "", "Source System")
dbutils.widgets.text("table_name", "", "Table Name")
dbutils.widgets.text("landing_timestamp", "", "Landing Timestamp")
dbutils.widgets.text("source_file_format", "", "Source File Format")

# Get widget values into variables
source_system = dbutils.widgets.get("source_system")
table_name = dbutils.widgets.get("table_name")
landing_timestamp = dbutils.widgets.get("landing_timestamp")
source_file_format = dbutils.widgets.get("source_file_format")

print(f"source_system: {source_system}")
print(f"table_name: {table_name}")
print(f"landing_timestamp: {landing_timestamp}")
print(f"source_file_format: {source_file_format}")

In [0]:
# Build the landing path and verify it exists before reading
landing_path = f"/Volumes/adb_caresync/landing/landing_volume/{source_system}/{table_name}/{landing_timestamp}"

try:
    path_exists = len(dbutils.fs.ls(landing_path)) > 0
except Exception:
    # dbutils.fs.ls raises if the path or any parent directory does not exist
    path_exists = False

# exit is called outside the try block so it is not swallowed by the except
if not path_exists:
    print(f"No new files received at {landing_path}. Exiting notebook.")
    dbutils.notebook.exit("No new files received")

print(f"Landing path found: {landing_path}")

In [0]:
if source_file_format == "csv":
    landing_df = spark.read.format("csv").option("header", "true").option("inferSchema", "true").load(landing_path)
else:
    landing_df = spark.read.format("parquet").load(landing_path)

In [0]:
from pyspark.sql.functions import lit, current_timestamp

updated_df = landing_df.withColumn("landing_timestamp", lit(landing_timestamp)) \
                       .withColumn("insert_timestamp", current_timestamp())

In [0]:
updated_df.write.mode("append").saveAsTable(f"adb_caresync.bronze.{table_name}")